# Week 8 – Day 3

# Machine Learning Modeling

## Objective

Train and evaluate three machine learning models using engineered stock market features.

### Models

- Logistic Regression
- Random Forest
- Gradient Boosting

### Validation

TimeSeriesSplit Cross Validation

### Goal

Identify the best-performing model and save it for deployment.

In [16]:

# Import Required Libraries


import pandas as pd
import numpy as np

from sklearn.model_selection import TimeSeriesSplit

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

import joblib

In [17]:

# Load Feature Dataset


data = pd.read_csv("../data/features.csv")

data.head()

,Date,Open,High,Low,Close,Volume,SMA_10,SMA_20,SMA_50,EMA_10,EMA_20,Return,Lag_1,Lag_5,Lag_21,Volatility_20,RSI,MACD,Signal
0,2019-01-01,498.077624,498.985034,491.371654,496.196381,9746670,NaN,NaN,NaN,496.196381,496.196381,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
1,2019-01-02,493.319235,498.852220,487.343625,489.733887,15628818,NaN,NaN,NaN,492.642009,492.803571,-0.013024,NaN,NaN,NaN,NaN,NaN,-0.144992,-0.080551
2,2019-01-03,490.220755,493.363478,482.518879,483.691864,16288287,NaN,NaN,NaN,489.044110,489.457807,-0.012337,-0.013024,NaN,NaN,NaN,NaN,-0.372111,-0.200043
3,2019-01-04,485.750155,488.870748,478.535167,486.303436,18516544,NaN,NaN,NaN,488.141180,488.547189,0.005399,-0.012337,NaN,NaN,NaN,NaN,-0.372000,-0.258294
4,2019-01-07,489.999414,495.067613,487.343578,489.003479,12060290,NaN,NaN,NaN,488.388723,488.657562,0.005552,0.005399,NaN,NaN,NaN,NaN,-0.256667,-0.257810


In [18]:
# Remove missing values

data = data.dropna()

data.shape

(1431, 19)

# Prepare Features and Target

In [19]:
import numpy as np

# Next day return
data["Target_Return"] = data["Close"].pct_change().shift(-1)

# Binary Target
data["Target"] = np.where(data["Target_Return"] > 0, 1, 0)

# Remove NaN rows
data.dropna(inplace=True)

# Check
print(data[["Close", "Target_Return", "Target"]].head())

         Close  Target_Return  Target
49  596.365234      -0.004268       0
50  593.820068      -0.014834       0
51  585.011597       0.021488       1
52  597.582458       0.019629       1
53  609.312439      -0.000799       0


In [20]:
feature_columns = [

    "SMA_10",
    "SMA_20",
    "SMA_50",

    "EMA_10",
    "EMA_20",

    "RSI",

    "MACD",

    "Lag_1",
    "Lag_5",
    "Lag_21"
]

X = data[feature_columns]

y = data["Target"].astype(int)

In [21]:
tscv = TimeSeriesSplit(
    n_splits=5
)

# Initialize Models

In [22]:
models = {

    "Logistic Regression":
        LogisticRegression(max_iter=1000),

    "Random Forest":
        RandomForestClassifier(
            random_state=42
        ),

    "Gradient Boosting":
        GradientBoostingClassifier(
            random_state=42
        )
}

# Train and Evaluate Models

In [23]:
results = []

best_score = 0

best_model = None

best_name = ""

for model_name, model in models.items():

    accuracy = []

    precision = []

    recall = []

    f1 = []

    for train_index, test_index in tscv.split(X):

        X_train = X.iloc[train_index]

        X_test = X.iloc[test_index]

        y_train = y.iloc[train_index]

        y_test = y.iloc[test_index]

        # Skip folds with one class
        if y_train.nunique() < 2:
            continue

        model.fit(
            X_train,
            y_train
        )

        prediction = model.predict(X_test)

        accuracy.append(
            accuracy_score(
                y_test,
                prediction
            )
        )

        precision.append(
            precision_score(
                y_test,
                prediction,
                zero_division=0
            )
        )

        recall.append(
            recall_score(
                y_test,
                prediction,
                zero_division=0
            )
        )

        f1.append(
            f1_score(
                y_test,
                prediction,
                zero_division=0
            )
        )

    mean_f1 = np.mean(f1)

    results.append({

        "Model": model_name,

        "Accuracy": np.mean(accuracy),

        "Precision": np.mean(precision),

        "Recall": np.mean(recall),

        "F1 Score": mean_f1

    })

    if mean_f1 > best_score:

        best_score = mean_f1

        best_model = model

        best_name = model_name

In [24]:
results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.495798,0.509619,0.735944,0.557659
1,Random Forest,0.503361,0.547552,0.383112,0.436484
2,Gradient Boosting,0.510924,0.652724,0.288046,0.356076


# Best Model

In [25]:
print("Best Model")

print(best_name)

print()

print("F1 Score")

print(best_score)

Best Model
Logistic Regression

F1 Score
0.5576593038825495


# Save Best Model

In [26]:
joblib.dump(

    best_model,

    "../models/best_model.joblib"

)

print("Best model saved successfully.")

Best model saved successfully.


In [27]:
results_df.to_csv(

    "../reports/model_results.csv",

    index=False

)

# Conclusion

Three machine learning models were trained using TimeSeriesSplit cross-validation.

Evaluation was based on:

- Accuracy
- Precision
- Recall
- F1 Score

The model with the highest average F1 Score was selected as the final model and saved as:

`models/best_model.joblib`

This model will be used during the final deployment and prediction stage of the capstone project.